In [ ]:
# Load Packages
import pandas
import os
import numpy as np
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import json
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from platform import python_version
from sklearn import __version__ as sklearn_version

# Get Package Versions
print(f"python version: {python_version()}")
print(f"pandas version: {pandas.__version__}")
print(f"os version: {os.name}")
print(f"numpy version: {np.__version__}")
#print(f"hail version: {hl.version()}") # If you get an error remove this line, Hail is loaded afterwords
# print(f"mpl_axes_aligner version: {mpl_axes_aligner.__version__}") has no version 
print(f"matplotlib version: {matplotlib.__version__}")
print(f"json version: {json.__version__}")
print(f"plotly version: {plotly.__version__}")
print(f"sklearn version: {sklearn_version}")

In [ ]:
# This query represents dataset "CF Carrier Population" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_44345198_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (40479565) 
                    AND is_standard = 1 )) criteria ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (45769146, 254320, 45768915, 441267, 193174, 4144583)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )"""

dataset_44345198_person_df = pandas.read_gbq(
    dataset_44345198_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_44345198_person_df.head(5)

In [ ]:
%env DATASET_44345198_VCF_DIR= # insert google cloud bucket with data here

In [ ]:
import os
import subprocess

# The extraction workflow outputs a manifest file upon completion.
manifest_file = os.environ['DATASET_44345198_VCF_DIR'] + '/manifest.txt'

assert subprocess.run(['gsutil', '-q', 'stat', manifest_file]).returncode == 0, (
  "!" * 100 + "\n\n" +
  "VCF extraction has not completed.\n" +
  "Please monitor the extraction sidepanel for completion before continuing.\n\n" +
  "!" * 100
)

print("VCF extraction has completed, continuing")


In [ ]:
# Confirm Spark is installed.
try:
    import pyspark
except ModuleNotFoundError:
    print("!" * 100 + "\n\n"
          "In the Researcher Workbench, Hail can only be used on a Dataproc cluster.\n"
          "Please use the 'Cloud Analysis Environment' side panel to update your runtime compute type.\n\n" +
          "!" * 100)

# Initialize Hail
import hail as hl
import os
from hail.plot import show

hl.init(default_reference='GRCh38', global_seed = 0)
hl.plot.output_notebook()

In [ ]:
# Create Hail Matrix table
# This can take a few hours for a dataset with hundreds of participants
workspace_bucket = os.environ['WORKSPACE_BUCKET']
vcf_dir = os.environ['DATASET_44345198_VCF_DIR']
hail_matrix_table_gcs = f'{workspace_bucket}/dataset_44345198.mt'
#hl.import_vcf(f'{vcf_dir}/*.vcf.gz', force_bgz=True, array_elements_required=False).write(hail_matrix_table_gcs)


In [ ]:
# Read Hail Matrix table
mt_44345198 = hl.read_matrix_table(hail_matrix_table_gcs)

In [ ]:
# Indicate the chromosomal location of each CFTR intron and extract variants belonging to each intron
# chromosomal locations based on GRCh38 genome assembly of the ENST00000003084.11 CFTR Transcript 

Intron_1 = ['chr7:117480147-117504252']
Intron_2 = ['chr7:117504363-117509033']
Intron_3 = ['chr7:117509142-117530898']
Intron_4 = ['chr7:117531114-117534275']
Intron_5 = ['chr7:117534365-117535247']
Intron_6 = ['chr7:117535411-117536547']
Intron_7 = ['chr7:117536673-117540099']
Intron_8 = ['chr7:117540346-117542015']
Intron_9 = ['chr7:117542108-117548640']
Intron_10 = ['chr7:117548823-117559463']
Intron_11 = ['chr7:117559655-117587738']
Intron_12 = ['chr7:117587833-117590352']
Intron_13 = ['chr7:117590439-117591933']
Intron_14 = ['chr7:117592657-117594929']
Intron_15 = ['chr7:117595058-117602825']
Intron_16 = ['chr7:117602863-117603531']
Intron_17 = ['chr7:117603782-117606673']
Intron_18 = ['chr7:117606753-117610518']
Intron_19 = ['chr7:117610669-117611580']
Intron_20 = ['chr7:117611808-117614612']
Intron_21 = ['chr7:117614713-117627521']
Intron_22 = ['chr7:117627770-117642437']
Intron_23 = ['chr7:117642593-117652841']
Intron_24 = ['chr7:117652931-117664687']
Intron_25 = ['chr7:117664860-117665458']
Intron_26 = ['chr7:117665564-117666907']

# Create a dictionary with this data
mt_dict = {}

for i in range(1, 27):
    filtered_mt = hl.filter_intervals(
        mt_44345198,
        [hl.parse_locus_interval(x)
         for x in globals()[f'Intron_{i}']]
    )
    mt_dict[f'mt{i}'] = filtered_mt

In [ ]:
# Load excel file with exonic CFTR Variant Data
excel_df = pandas.read_excel(f'{workspace_bucket}/data/CFTR2 variants genomic locations_2023.xlsx')

# Code to extract the starting position of the first CFTR variant and the ending position of the last CFTR variant
# This allows us to have a targetted region from which we will query the variants that individuals in the 
# cohort have
start_pos = min(excel_df['grch38_pos'])
start_pos_chr = min(excel_df['grch38_chr'])
end_pos = max(excel_df['grch38_pos'])+ 1 # 1 because last variant is SNP, change accordingly (e.g if the alt
# allele changed 3 nucleotides then + 3)
end_pos_chr = max(excel_df['grch38_chr'])

# Extract Variants from selected region
region_to_analyze = [f'chr{start_pos_chr}:{start_pos}-chr{end_pos_chr}:{end_pos}']
mt_annotate = hl.filter_intervals(
    mt_44345198,
    [hl.parse_locus_interval(x,)
     for x in region_to_analyze])

In [ ]:
# Collecting Relatedness Info
auxiliary_path = "gs://fc-aou-datasets-controlled/v7/wgs/short_read/snpindel/aux"
relatedness_path= f'{auxiliary_path}/relatedness/relatedness.tsv'
relatedness=hl.import_table(relatedness_path)
relatedness_df = relatedness.to_pandas()
people = mt_annotate.col.s.collect()

# Collecting individuals which are related to one another (kinship score >= 0.125) , Started with n = 1574
relatedness_df = relatedness_df[relatedness_df['kin'].astype(float) >= 0.125]
filtered_relatedness_df = relatedness_df[(relatedness_df['i.s'].isin(people)) & (relatedness_df['j.s'].isin(people))]
filtered_relatedness_df # 13 pairs of related individuals, remove one of the individuals from each pair 

# Removing data (across all introns) for one of the individual in each related pairs (n = 1561 remaining)
for x in filtered_relatedness_df['i.s']:
    mt_annotate = mt_annotate.filter_cols(mt_annotate.s != x) 

# Uploading Hail Matrix Table GT values (all introns), downloading and reading them in as a pandas dataframe 
mt_annotate.GT.export(f'{workspace_bucket}/data/annotateGT.tsv')
df_annotations = pandas.read_csv(f'{workspace_bucket}/data/annotateGT.tsv', sep= '\t')


## Annotate individuals within the cohort by their CFTR variant(s) and their deemed CF Status

In [ ]:
## Do not need to run again if results have been uploaded 
import json
# create cdna column in df_annotations 
df_annotations['alleles'] = df_annotations['alleles'].apply(lambda x: json.loads(x))

df_annotations['cdna_name']= ""
df_annotations['legacy_name']= ""
df_annotations['cftr2_annotation']= ""

excel_drop = []

# Filter excel sheet so it only contains variants found in the individual data (not really necessary)
for x_index, x_row in excel_df.iterrows():
    match = False 
    for y_index, y_row in df_annotations.iterrows():
        if ((f"chr{x_row['grch38_chr']}:{x_row['grch38_pos']}" == y_row['locus']) & 
            (str(x_row['grch38_ref']) == (y_row['alleles'])[0]) &
                (str(x_row['grch38_alt']) == (y_row['alleles'])[1])):
            match = True
            break
    if not match:
        excel_drop.append(x_index)

excel_df.drop(excel_drop, inplace = True)

variant_drop = []

# Filtering individual data so it only contains variants in the excel sheet 
for y_index, y_row in df_annotations.iterrows():
    match = False
    for x_index, x_row in excel_df.iterrows():
        if ((f"chr{x_row['grch38_chr']}:{x_row['grch38_pos']}" == y_row['locus']) & 
            (str(x_row['grch38_ref']) == (y_row['alleles'][0])) &
                (str(x_row['grch38_alt']) == y_row['alleles'][1])):
            
            df_annotations.at[y_index,'cdna_name'] = excel_df.at[x_index,'cdna_name']
            df_annotations.at[y_index,'legacy_name'] = excel_df.at[x_index,'all_legacy_names']
            df_annotations.at[y_index, 'cftr2_annotation'] = excel_df.at[x_index, 'cftr2_annotation']
            
            match = True
            break 
                
    if not match:
        variant_drop.append(y_index)

df_annotations.drop(variant_drop, inplace = True) 
df_annotations2 = df_annotations.copy(deep=True)

# Replacing call values with annotation for the presence or lack of mutation 
df_annotations = df_annotations.replace('0/0', "No Mutation")
df_annotations = df_annotations.replace('0/1',"Heterozygous Mutation")
df_annotations = df_annotations.replace('1/0',"Heterozygous Mutation")
df_annotations = df_annotations.replace('1/1',"Homozygous Mutation")

# Upload files online
df_annotations.to_csv(f'{workspace_bucket}/data/df_annotations3.csv', index=False)
df_annotations2.to_csv(f'{workspace_bucket}/data/df_annotations4.csv', index=False)

In [ ]:
# Download and manipulate the data frame to be more presentable

df_annotations2 = pandas.read_csv(f'{workspace_bucket}/data/df_annotations4.csv')
df_annotations = pandas.read_csv(f'{workspace_bucket}/data/df_annotations3.csv')
df_annotations.insert(2,"cdna_name",df_annotations.pop("cdna_name"))
df_annotations.insert(3,"legacy_name",df_annotations.pop("legacy_name"))
df_annotations.insert(4,"cftr2_annotation",df_annotations.pop("cftr2_annotation"))

# Calculate summary statistics of each variant in the cohort
df_annotation_summary = df_annotations.iloc[:,0:4]
df_annotation_summary["Homozygous Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("Homozygous Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["Heterozygous Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("Heterozygous Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["No Mutation %"] = df_annotations.apply(lambda row: 
                                                                      (row.str.count("No Mutation").sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)
df_annotation_summary["NaN %"] = df_annotations.apply(lambda row: 
                                                                      (row.isna().sum()/
                                                                       (df_annotations.shape[1]-5))*100, axis=1)

# Export Variant annotation 
df_annotation_summary.to_excel('Total Variant Annotation.xlsx')

In [ ]:
# Create a plot of the annotation results 

fig, ax = plt.subplots(figsize=(24,12))
y1 = df_annotation_summary["No Mutation %"]
y2 = df_annotation_summary["Heterozygous Mutation %"]
y3 = df_annotation_summary["Homozygous Mutation %"]
y4 = df_annotation_summary["NaN %"]

ax.bar(df_annotation_summary["legacy_name"], y1, color= 'beige', edgecolor= 'black', label="No Mutation %")

ax.bar(df_annotation_summary["legacy_name"], y2, bottom = y1, color= 'violet', edgecolor = 'black', 
       label="Heterozygous Mutation %")

ax.bar(df_annotation_summary["legacy_name"],df_annotation_summary["Homozygous Mutation %"], bottom = y1 + y2,
       color= 'black', edgecolor = 'black', label="Homozygous Mutation %")

ax.bar(df_annotation_summary["legacy_name"],df_annotation_summary["NaN %"],bottom = y1 + y2 + y3,
       color= 'red', edgecolor = 'black', label="NaN %")

ax.set_xlabel("Legacy Variant Name")
ax.set_ylabel("%")
ax.set_title("Variant Profiling of the n=307 pwCF Cohort")
ax.legend(loc = 'lower right')
plt.xticks(rotation=90)
size = 12
params = {'legend.fontsize': 'large',
          'figure.figsize': (20,8),
          'axes.labelsize': size*1.25,
          'axes.titlesize': size*1.25,
          'xtick.labelsize': size*0.75,
          'ytick.labelsize': size*0.75,
          'axes.titlepad': 15}
plt.rcParams.update(params)
plt.show()


In [ ]:
# Assess the top 4 mutations in greater detail

# Filter for the top 4 mutations
df_annotations_cropped = df_annotations[(df_annotations["legacy_name"] == "125G/C|5UTR-8G->C") |
                                        (df_annotations["legacy_name"] == "T854T|2694T/G|T854T (2694T/G)") |
                                        (df_annotations["legacy_name"] == "F508del|1653delCTT|[delta]F508|^F508|dF508") |
                                        (df_annotations["legacy_name"] ==  "V470M")]

# Make a table for annotation purposes 
df_annotations_cropped_transposed = df_annotations_cropped.iloc[:,4:].transpose().copy(deep=True)
df_annotations_cropped_transposed.columns = ["125G/C|5UTR-8G->C","V470M","F508del|1653delCTT|[delta]F508|^F508|dF508",
                                            "T854T|2694T/G|T854T (2694T/G)"]
df_annotations_cropped_transposed = df_annotations_cropped_transposed[1:].reset_index(drop=True)
df_annotations_cropped_transposed = df_annotations_cropped_transposed.replace({'Homozygous Mutation': 2, 
                                                                              'Heterozygous Mutation': 1,
                                                                              'No Mutation': 0})
#df_annotations_cropped_transposed = df_annotations_cropped_transposed.fillna("None")


## Extract Genotype Call Data based on Intronic Region and perform Quality Control

In [ ]:
# Choose Intronic Region to assess (below you can change )
intron=26
mt = mt_dict['mt26']

# Collecting Relatedness Info
relatedness=hl.import_table(relatedness_path)
relatedness_df = relatedness.to_pandas()
people = mt.col.s.collect()

# Collecting individuals which are related to one another (no filter in terms of relatedness score set)
relatedness_df = relatedness_df[relatedness_df['kin'].astype(float) >= 0.125]
filtered_relatedness_df = relatedness_df[(relatedness_df['i.s'].isin(people)) & (relatedness_df['j.s'].isin(people))]
print(filtered_relatedness_df)                                

# Removing data (in chosen intron) for one of the individuals in each related pairs 
for x in filtered_relatedness_df['i.s']:
    mt = mt.filter_cols(mt.s != x)    

# Check details of the variants found within each intron, in greater detail
hl.summarize_variants(mt)


In [ ]:
# Split Multi-Allelic Loci into multiple Bi-Allelic Loci 
mt = hl.split_multi_hts(mt)

hl.summarize_variants(mt)

# Convert entries with a genotype quality of below 20 to NA
mt_filtered = mt.annotate_entries(GQ=hl.if_else((mt.GQ < 20), hl.missing(mt.GQ.dtype), mt.GQ))

# Remove rows which have a genotype quality of NA in over 10% of columns 
cond = hl.agg.count_where(hl.is_missing(mt_filtered.GQ))
mt_filtered2 = mt_filtered.filter_rows((cond/mt_filtered.count_cols()) <= 0.1)

# check how many variants remain
hl.summarize_variants(mt_filtered2)

In [ ]:
# Annotate variants with major-minor allele frequencies, etc. 
mt_filtered2=hl.variant_qc(mt_filtered2)

# Filter rows where the dominant allele is present less than 99b% of the time or minor alleles is present more than
# 1% of the time (lower standards because F508del make less portion of samples here than in pwCF)
mt_filtered2 = mt_filtered2.filter_rows(mt_filtered2.variant_qc.AF[1] > 0.01)

# Check allele frequencies of remaining variants
mt_filtered2.variant_qc.AF[1].show()

# check how many variants remain
hl.summarize_variants(mt_filtered2)

In [ ]:
# Create a column which indicates how many GT values are defined vs. missing in the form [missing, defined], 
# for each row, for PCA it is important to make sure there are no missing values
mt_filtered2 = mt_filtered2.annotate_rows(counter=[hl.agg.sum(hl.is_missing(mt_filtered2.GT)) , hl.agg.sum
                                                              (hl.is_defined(mt_filtered2.GT))])

In [ ]:
# Turn call values into integers
# Let the integer be an average of the call values from both chromosomes. E.g: 0/0 = 0, 0/1 and 1/0 = 0.5, 1/1 =1 etc. 
# All missing values are temporarily given the the value: 0
def append_genotype_call(genotype):
    genotype = hl.str(genotype).split('/')
    left_side = hl.int32(genotype[0])
    right_side = hl.int32(genotype[1])
    
    geno = (left_side+right_side)/2
    
    return hl.float(geno)

mt_filtered2 = mt_filtered2.annotate_entries(intermediary_GT=hl.or_else(append_genotype_call(mt_filtered2.GT), 
                                                                 hl.float(0)))


In [ ]:
# Sum all the values in the row, so that the mean of the row can be calculated
mt_filtered2 = mt_filtered2.annotate_rows(sum_GT=hl.agg.sum(mt_filtered2.intermediary_GT))

#Determine the mean of the row by dividing the sum of the row with the amount of defined values in a row
def impute_call(genotype):
    genotype = mt_filtered2.sum_GT/(mt_filtered2.counter[1])
    return hl.float(genotype)

# Instead of setting missing values to 0, set the missing values of each row to the calculated mean of the row
mt_filtered3 = mt_filtered2.annotate_entries(intermediary_GT2=hl.or_else(append_genotype_call(mt_filtered2.GT)
                                                                   ,impute_call(mt_filtered2.GT)))

# Create a parallel dataframe which still has missing values, which will be used for machine learning (ML),
# as XGBoost machine learning is able to adjust to missing values (unlike PCA)
mt_filtered4 = mt_filtered2.annotate_entries(ML=append_genotype_call(mt_filtered2.GT))

In [ ]:
# Upload Matrix table as TSV (and also keep one as reference)
mt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/gt2.tsv')
mt_filtered4.ML.export(f'{workspace_bucket}/data/gt3.tsv')
mt_filtered4.alleles.export(f'{workspace_bucket}/data/alleles.tsv')
#bmt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/index_reference.tsv') # already performed, do not do it again


# Download TSV as Pandas Data Frame
df_intermediate = pandas.read_csv(f'{workspace_bucket}/data/gt2.tsv', sep= '\t')
ML_file = pandas.read_csv(f'{workspace_bucket}/data/gt3.tsv', sep= '\t')
Alleles = pandas.read_csv(f'{workspace_bucket}/data/alleles.tsv', sep= '\t')
index_reference = pandas.read_csv(f'{workspace_bucket}/data/index_reference.tsv', sep= '\t')

In [ ]:
# Upload Matrix table as TSV (and also keep one as reference)
mt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/gt2.tsv')
mt_filtered4.ML.export(f'{workspace_bucket}/data/gt3.tsv')
mt_filtered4.alleles.export(f'{workspace_bucket}/data/alleles.tsv')
#bmt_filtered3.intermediary_GT2.export(f'{workspace_bucket}/data/index_reference.tsv') # already performed, do not do it again

# Download TSV as Pandas Data Frame
df_intermediate = pandas.read_csv(f'{workspace_bucket}/data/gt2.tsv', sep= '\t')
ML_file = pandas.read_csv(f'{workspace_bucket}/data/gt3.tsv', sep= '\t')
ML_drop = ML_file.drop('alleles',axis=1)
Alleles = pandas.read_csv(f'{workspace_bucket}/data/alleles.tsv', sep= '\t')

# Set Locus column as index
ML_final = ML_drop.set_index('locus')

In [ ]:
ML_final = ML_final.transpose().reset_index(drop=True)
ML_final['F508del'] = ''
ML_final['F508del'] = df_annotations_cropped_transposed['F508del|1653delCTT|[delta]F508|^F508|dF508']
ML_final['V470M'] = ''
ML_final['V470M'] = df_annotations_cropped_transposed['V470M']

ML_final.to_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/Intron_{intron}_CF_Carriers.xlsx', index=False)
Alleles.to_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/Intron_{intron}_Alleles_CF_Carriers.xlsx', index=False)

### Edit Allele Names, upload and use code to automatically adjust Allele names for genotype call data

In [ ]:
import pandas as pd

introns = [i for i in range(1, 27) if i != 16]
for x in introns:
           table = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{x}_CF_Carriers.xlsx')
           table['F508del'] = df_annotations_cropped_transposed['F508del|1653delCTT|[delta]F508|^F508|dF508']
           table['V470M'] = df_annotations_cropped_transposed['V470M']
           table.to_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{x}_CF_Carriers.xlsx', index=False)

# Not necessary to repeat if already done once 

In [ ]:
# Upload allele files to workbucket (after adjusting consistency locus names to be consistent with those within CF carriers variants)
introns = [i for i in range(1, 27) if i != 16] # Create list of introns from 1 to 26, not including 16

for x in introns:
    new_table = pd.read_excel(f'Intron_{x}_Alleles_CF_Carriers.xlsx')
    new_table.to_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/Intron_{x}_Alleles_CF_Carriers.xlsx', index=False)

In [ ]:
# Adjust genotype call files

for x in introns: 
    allele_table = pd.read_excel(f'Intron_{x}_Alleles_CF_Carriers.xlsx') # load alleles file
    snps_table = pd.read_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/Intron_{x}_CF_Carriers.xlsx') # load snp genotype call file
    snps = allele_table['locus'] # load correct snp locus names
    if snps.is_unique: # only proceeds if all snps are unique, personal coherency check 
        new_columns = snps.tolist()+ snps_table.columns[-2:].tolist() # Adding F508del, V470M and F508del + V470M column
        snps_table.columns = new_columns
        snps_table.columns
        snps_table.to_excel(f'{workspace_bucket}/data/Genotype Files/CF Carriers/adjusted/Intron_{x}_CF_Carriers.xlsx', index=False)
    else:
        print(f'error: values are not unique in intron {x}')

In [ ]:
import session_info
session_info.show(_)